# Figure 5. Cooperation modes and predicted five-year conflict counts

## Data sources and analysis inputs

- **Historical data**. The historical event records (1951–2019) are from Kåresdotter et al. (2023), available on [Zenodo](https://doi.org/10.5281/zenodo.7465153). The analysis uses a prepared country-year panel (`panel.csv`) that combines the event records with cooperation-mode indicators, the subsequent five-year conflict count, and contextual covariates.
- **Future data**. SSP socioeconomic projections are available from the [IIASA SSP Scenario Database](https://iiasa.ac.at/models-tools-data/ssp). This notebook uses the prepared 2050 SSP5 covariate table (`ssp5_2050.csv`), which combines socioeconomic and hydroclimatic inputs.

In [ ]:
from pathlib import Path

PANEL_FILE = Path("panel.csv")
SSP5_FILE = Path("ssp5_2050.csv")
FIGURE5_OUTPUT_DIR = Path("outputs") / "figure05"
FIGURE5_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## A. Historical prediction contrasts and SHAP (training through 2010)

In [ ]:
# Data preparation.
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
df = pd.read_csv(PANEL_FILE, float_precision="round_trip")
label = ['Cross-Border Basin Agreements','Collaborative Planning and Adaptation Strategies','Joint Water Allocation Models',
         'Joint Data-Sharing Systems','Transboundary Water Quality Standards','Coordinated Hydropower Operations']
df.loc[df['Both'] == 1, 'Conflict'] = 1
df['potential_Conflict'] = df['Conflict']
SEdata = ['gdp_percapita', 'export', 'population_density', 'rural_population', 'precipitation', 'bws_score']
df = df[['year', 'ccode_iso3', 'Country_count', 'Conflict', 'potential_Conflict'] + SEdata + label]
pd.set_option('future.no_silent_downcasting', True)
df = df.replace('..', np.nan)
scaler = StandardScaler()
df[SEdata] = scaler.fit_transform(df[SEdata])
df = df.dropna(subset=df.columns.difference(['Country_count']))
num_rows = df.shape[0]
print("Number of rows:", num_rows)

In [ ]:
# Train through 2010; test later conflict-only rows.
y = 'Country_count'
x = df[['Conflict'] + SEdata + label]
mask_train = df['year'] <= 2010
x_train = x[mask_train]
y_train = df[y][mask_train]
#mask_test = df['year'] > 2010
#x_test = x[mask_test]
#y_test = df[y][mask_test]
mask_year = df['year'] > 2010
mask_conflict = df['potential_Conflict'] == 1
mask_test = mask_year & mask_conflict
x_test = x[mask_test]
y_test = df[y][mask_test]

In [ ]:
# Prediction contrasts; Conflict changes along with mode indicators.
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
rfr = RandomForestRegressor(random_state=42)
rfr.fit(x_train, y_train)
x_test_case1 = x_test.copy()
x_test_case1[label] = 0
y_pred_case1 = rfr.predict(x_test_case1)
df_copy = df.copy()
df_copy.loc[mask_test, 'Noncooperation'] = y_pred_case1
x_test_case2 = x_test.copy()
x_test_case2['Conflict'] = 0
y_pred_case2 = rfr.predict(x_test_case2)
df_copy.loc[mask_test, 'Cooperation'] = y_pred_case2
df_copy.loc[mask_test, 'Combined Mitigation'] = df_copy.loc[mask_test, 'Cooperation'] - df_copy.loc[mask_test, 'Noncooperation']
print(df_copy[mask_test][['year', 'ccode_iso3', 'Noncooperation', 'Cooperation', 'Combined Mitigation']])

In [ ]:
# Calculate the three-mode contrast and row-based 95% t interval for Figure 5f.
from scipy import stats

integrative_modes = [
    'Cross-Border Basin Agreements',
    'Collaborative Planning and Adaptation Strategies',
    'Joint Water Allocation Models',
]
observed_combination = x_test[integrative_modes].eq(1).all(axis=1)
observed_test = x_test.loc[observed_combination]
if len(observed_test) < 2:
    raise ValueError('At least two test observations with all three modes are required.')

# Compare the fixed three-mode coding with the paired conflict-only baseline.
combination_case = observed_test.copy()
combination_case['Conflict'] = 0
combination_case[label] = 0
combination_case[integrative_modes] = 1
baseline_case = x_test_case1.loc[observed_combination]
three_mode_differences = rfr.predict(combination_case) - rfr.predict(baseline_case)

three_mode_mean = float(three_mode_differences.mean())
three_mode_half_width = stats.t.ppf(0.975, len(three_mode_differences) - 1) * stats.sem(three_mode_differences)
three_mode_summary = pd.Series({
    'Observations': len(three_mode_differences),
    'Mean predicted difference': three_mode_mean,
    'Lower 95% bound': three_mode_mean - three_mode_half_width,
    'Upper 95% bound': three_mode_mean + three_mode_half_width,
})
print(three_mode_summary)

In [ ]:
# Plot the calculated three-mode contrast and 95% interval.
import matplotlib.pyplot as plt

effect = three_mode_summary['Mean predicted difference']
ci_low = three_mode_summary['Lower 95% bound']
ci_high = three_mode_summary['Upper 95% bound']

fig, ax = plt.subplots(figsize=(2.5, 1))

ax.axvline(x=0, color='#888888', linestyle='--', linewidth=1.4)

ax.errorbar(
    x=effect,
    y=0,
    xerr=[[effect - ci_low],
          [ci_high - effect]],
    fmt='s',
    markersize=7,
    color='#115757',
    ecolor='#115757',
    elinewidth=1.5,
    capsize=4,
    capthick=1.5
)

ax.set_yticks([])
#ax.set_xlabel("Mitigation Effect")

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)


left, right = min(ci_low, 0), max(ci_high, 0)
padding = max((right - left) * 0.08, 0.05)
ax.set_xlim(left - padding, right + padding)
ax.spines['bottom'].set_linewidth(1.0)
ax.spines['bottom'].set_color('#000000')
ax.set_xticklabels([])

ax.set_ylim(-0.5, 0.5)

plt.tight_layout()


fig.savefig(FIGURE5_OUTPUT_DIR / 'figure05f_three_mode_contrast.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Single-mode prediction contrasts.
results_mean = {}
results_diff = {}
for l in label:
    x_test_case4 = x_test.copy()
    x_test_case4['Conflict'] = 0
    x_test_case4[label] = 0
    x_test_case4[l] = 1
    y_pred_case = rfr.predict(x_test_case4)
    diff = y_pred_case - y_pred_case1
    results_diff[l] = diff.tolist()
    diff_mean = np.mean(diff)
    results_mean[l] = diff_mean
mask_test = mask_test.reindex(df.index, fill_value=False)
for l, diff_list in results_diff.items():
    df[f'Cooperation_Diff_{l}'] = np.nan
    df.loc[mask_test, f'Cooperation_Diff_{l}'] = diff_list
df_mask_test = df[mask_test]
#print(df_mask_test[['year', 'ccode_iso3'] + [col for col in df_mask_test.columns if 'Cooperation_Diff' in col]])

results_mean_df = pd.DataFrame(list(results_mean.items()), columns=['Variable', 'Mitigating Effect'])
print(results_mean_df)

In [ ]:
# Row-based t intervals.
import numpy as np
import pandas as pd
from scipy import stats
results_ci = {}
for l in label:
    diff = np.array(results_diff[l])
    diff_mean = results_mean[l]
    std_err = np.std(diff, ddof=1) / np.sqrt(len(diff))
    h = stats.t.ppf((1 + 0.95) / 2., len(diff) - 1)
    ci_lower = diff_mean - h * std_err
    ci_upper = diff_mean + h * std_err
    results_ci[l] = (ci_lower, ci_upper)
results_ci_df = pd.DataFrame(list(results_ci.items()), columns=['Variable', '95% CI'])
results_ci_df['Lower Bound'] = results_ci_df['95% CI'].apply(lambda x: x[0])
results_ci_df['Upper Bound'] = results_ci_df['95% CI'].apply(lambda x: x[1])
results_ci_df.drop(columns=['95% CI'], inplace=True)
print(results_ci_df)

In [ ]:
# Plot the single-mode contrasts and intervals displayed in Figure 5e.
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 8
plot_data = results_mean_df.merge(results_ci_df, on='Variable', validate='one_to_one')
plot_data = plot_data.set_index('Variable').loc[label]
scores = plot_data['Mitigating Effect'].tolist()
reference_values1 = plot_data['Lower Bound'].tolist()
reference_values2 = plot_data['Upper Bound'].tolist()
plt.figure(figsize=(3, 5))
plt.plot(scores, range(len(scores)), 'o-', color='#007878', markersize=11, linewidth=1, label='Scores')
plt.plot(reference_values1, range(len(reference_values1)), 'o-', color='#BDD7EE', markersize=9, linewidth=1, label='Reference 1')   # 5F93BB
plt.plot(reference_values2, range(len(reference_values2)), 'o-', color='#BDD7EE', markersize=9, linewidth=1, label='Reference 2')
plt.gca().invert_yaxis()
plt.xlim(min(reference_values1) - 0.1, max(reference_values2) + 0.5)
plt.yticks(range(len(scores)), label)
#for i, score in enumerate(scores):
#for i, value in enumerate(reference_values1):
#    plt.text(value - 0.1, i, f'{value:.4f}', va='center', color='#5F93BB')
#for i, value in enumerate(reference_values2):
#    plt.text(value + 0.1, i, f'{value:.4f}', va='center', color='#9AC5E7')
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)
ax.spines['left'].set_visible(True)
ax.set_xticks([])
plt.savefig(
    str(FIGURE5_OUTPUT_DIR / 'historical_line_0.png'),
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
# SHAP summary displayed in Figure 5d.
import shap
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

explainer = shap.Explainer(rfr, x_train)
shap_values = explainer(x_test)

custom_cmap = LinearSegmentedColormap.from_list(
    "custom_shap",
    ["#F5DF8B", "#007878"]
)

x_display = x_test[label]

shap.summary_plot(
    shap_values[:, x_test.columns.get_indexer(label)],
    x_display,
    show=False,
    cmap=custom_cmap
)

fig, ax = plt.gcf(), plt.gca()

ax.tick_params(axis='both', which='major', labelsize=8)

ax.set_facecolor('#FAFAFA')

for col in ax.collections:
    col.set_sizes([50])
    col.set_edgecolor("#2b2b2b")
    col.set_linewidth(0.5)
    col.set_alpha(0.99)

ax.axvline(0, color='gray', linewidth=0.8)

fig.savefig(
    str(FIGURE5_OUTPUT_DIR / 'historical_shap_summary.png'),
    dpi=600,
    bbox_inches='tight'
)
plt.show()

In [ ]:
# Mode/context correlation heatmap.
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
#correlation_matrix = df[SEdata + label].corr()
correlation_matrix = df[df['year'] <= 2019][SEdata + label].corr()
correlation_matrix = pd.DataFrame(correlation_matrix, index=label, columns=SEdata)


from matplotlib.colors import ListedColormap
discrete_hex_colors = [
    '#E3F8A6',
    '#BEE48C',
    '#9DD19E',
    '#5BB5B4',
    '#2773A7'
]

def hex_to_rgb_norm(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))

colors_normalized = [hex_to_rgb_norm(c) for c in discrete_hex_colors]

cmap = ListedColormap(colors_normalized)

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
plt.figure(figsize=(6, 5))
ax = sns.heatmap(correlation_matrix, cmap=cmap, annot=False, fmt=".3f", xticklabels=True, yticklabels=True, square=True,
            cbar_kws={"shrink": .6}, linewidths=1, vmin=-0.25, vmax=0.1)
cbar = ax.collections[0].colorbar
cbar.format = "%.3f"
vmin = cbar.vmin
vmax = cbar.vmax
vmid = (vmin + vmax) / 2
cbar.set_ticks([vmin, vmid, vmax])

plt.xticks(rotation=30, ha='right')
plt.yticks(va='center')
plt.tight_layout()
#    str(FIGURE5_OUTPUT_DIR / 'historical_correlation_matrix.png'),
#    bbox_inches='tight'
#)

plt.savefig(FIGURE5_OUTPUT_DIR / 'figure05b_mode_context.png', dpi=600, bbox_inches='tight')

In [ ]:
# Mode/mode correlation heatmap.

correlation_matrix = df[df['year'] <= 2019][label].corr()
mask = np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
plt.figure(figsize=(6, 5))
ax = sns.heatmap(correlation_matrix, cmap=cmap, annot=False, fmt=".3f", xticklabels=True, yticklabels=True, square=True,
            cbar_kws={"shrink": .6}, linewidths=1, vmin=0.1, vmax=1, mask=mask)
cbar = ax.collections[0].colorbar
cbar.format = "%.3f"
vmin = cbar.vmin
vmax = cbar.vmax
vmid = (vmin + vmax) / 2
cbar.set_ticks([vmin, vmid, vmax])
plt.xticks(rotation=30, ha='right')
plt.yticks(va='center')
plt.tight_layout()
#    str(FIGURE5_OUTPUT_DIR / 'historical_correlation_matrix2.png'),
#    bbox_inches='tight'
#)

plt.savefig(FIGURE5_OUTPUT_DIR / 'figure05b_mode_mode.png', dpi=600, bbox_inches='tight')

## B. Historical-model feature importance (training through 2019)

In [ ]:
# Read historical and SSP5 data.
import pandas as pd
df1 = pd.read_csv(PANEL_FILE, float_precision="round_trip")
df2 = pd.read_csv(SSP5_FILE, float_precision="round_trip")
df2['year'] = 2050
label = ['Joint Data-Sharing Systems','Cross-Border Basin Agreements','Transboundary Water Quality Standards',
         'Coordinated Hydropower Operations','Joint Water Allocation Models','Collaborative Planning and Adaptation Strategies']
#df3_labels = df3[['country_code'] + label]
#df2 = pd.merge(df2, df3_labels, on='country_code', how='left')

In [ ]:
# Append the 2050 covariates used in combined-data standardization.
df = pd.concat([df1, df2], ignore_index=True)

In [ ]:
# Preprocessing and combined-data standardization.
import numpy as np
from sklearn.preprocessing import StandardScaler
df.loc[df['Both'] == 1, 'Conflict'] = 1
SEdata = ['gdp_percapita', 'export', 'population_density', 'rural_population', 'precipitation', 'bws_score']
df = df[['year', 'ccode_iso3', 'Country_count', 'Conflict'] + SEdata + label]
pd.set_option('future.no_silent_downcasting', True)
df = df.replace('..', np.nan)
scaler = StandardScaler()
df[SEdata] = scaler.fit_transform(df[SEdata])
df = df.dropna(subset=df.columns.difference(['Country_count']))
num_rows = df.shape[0]
print("Number of rows:", num_rows)

In [ ]:
# Train the feature-importance model through 2019.
y = 'Country_count'
x = df[['Conflict'] + SEdata + label]
mask_train = df['year'] <= 2019
x_train = x[mask_train]
y_train = df[y][mask_train]

In [ ]:
# Historical-model importance; error bars show tree dispersion.
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
rfr = RandomForestRegressor(random_state=42)
rfr.fit(x_train, y_train)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
importances = rfr.feature_importances_
feature_importances = dict(zip(list(x_train.columns), importances))
sorted_feature_importances = sorted(feature_importances.items(), key=lambda item: item[1], reverse=True)
print("\nFeature names and importance:")
for feature, importance in sorted_feature_importances:
    print(f"{feature}: {importance:.4f}")
std = np.std([tree.feature_importances_ for tree in rfr.estimators_], axis=0)
sedata_length = len(SEdata)
label_indices = np.arange(sedata_length + 1, sedata_length + 1 + len(label))
label_importances = importances[label_indices]
label_importances_std = std[label_indices]
sorted_indices = np.argsort(label_importances)[::-1]
sorted_label_importances = label_importances[sorted_indices]
sorted_label_importances_std = label_importances_std[sorted_indices]
sorted_labels = [label[i] for i in sorted_indices]
color = "#72B3C5"
#color = "#438EBD"
plt.figure(figsize=(4.3, 6))
plt.title("Feature Importances")
plt.bar(range(len(sorted_labels)), sorted_label_importances, color=color, align="center",
        yerr=sorted_label_importances_std, capsize=5, width=0.8)
ax = plt.gca()
ax.margins(y=0.02)
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1.0)
plt.tick_params(axis='both', which='major', length=6, width=1.2)
plt.xticks(range(len(sorted_labels)), sorted_labels, rotation=30)
plt.xlim([-1, len(sorted_labels)])
plt.tight_layout()
plt.savefig(str(FIGURE5_OUTPUT_DIR / 'importance_Feature Importances3.png'), format='png', dpi=300, bbox_inches='tight')

In [ ]:
# All-feature importance table.
import pandas as pd

all_features = ['Conflict'] + SEdata + label
importances_all = importances

importance_sum = importances_all.sum()
importance_percent = importances_all / importance_sum * 100

feature_type = [
    'Policy' if f in label else 'Other'
    for f in all_features
]

df_importance = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances_all,
    'Percentage (%)': importance_percent,
    'Type': feature_type
})

df_importance = df_importance.sort_values(by='Importance', ascending=False)

df_importance['Importance'] = df_importance['Importance'].round(4)
df_importance['Percentage (%)'] = df_importance['Percentage (%)'].round(2)

print(df_importance)

In [ ]:
# SE/ENV/CC importance totals.
SE = {'population_density', 'gdp_percapita', 'rural_population', 'export'}
ENV = {'precipitation', 'bws_score'}
CC = ['Conflict'] + label

import pandas as pd

all_features = ['Conflict'] + SEdata + label
importances_all = importances

importance_sum = importances_all.sum()
importance_percent = importances_all / importance_sum * 100

def get_feature_type(f):
    if f in SE:
        return 'SE'
    elif f in ENV:
        return 'ENV'
    elif f in CC:
        return 'CC'
    else:
        return 'Other'

feature_type = [get_feature_type(f) for f in all_features]

df_importance = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances_all,
    'Percentage (%)': importance_percent,
    'Type': feature_type
})

df_importance = df_importance.sort_values(by='Importance', ascending=False)

df_grouped = df_importance.groupby('Type')['Percentage (%)'].sum().reset_index()
df_grouped = df_grouped.sort_values(by='Percentage (%)', ascending=False)

df_importance['Percentage (%)'] = df_importance['Percentage (%)'].round(6)
df_importance['Importance'] = df_importance['Importance'].round(6)

df_grouped['Percentage (%)'] = df_grouped['Percentage (%)'].round(6)

print("===== Individual feature importance =====")
print(df_importance)

print("\n===== Importance percentages by category=====")
print(df_grouped)

print("\nTotal category percentage:", df_grouped['Percentage (%)'].sum().round(6))

In [ ]:
# Radial plot: transformed angular widths and rank-based radius.
import numpy as np
import matplotlib.pyplot as plt

def ensure_list(x):
    if isinstance(x, str):
        return x.replace(',', ' ').split()
    return x

SEdata = ensure_list(SEdata)
label = ensure_list(label)

all_features = list(x_train.columns)
all_features = [f.strip() for f in all_features]

values = np.array(importances)
errors = np.array(std)

assert len(all_features) == len(values)

sorted_idx = np.argsort(values)[::-1]

all_features = [all_features[i] for i in sorted_idx]
values = values[sorted_idx]

SE = {'population_density', 'gdp_percapita', 'rural_population', 'export'}
ENV = {'precipitation', 'bws_score'}
GOV = set(label)

color_map = []
for f in all_features:
    if f in SE:
        color_map.append("#007878")
    elif f in ENV:
        color_map.append("#97D5CB")
    elif f in GOV:
        color_map.append("#F5DF8B")
    else:
        color_map.append("#F5DF8B")

inner_colors = []
for i in range(len(values)):
    if i % 2 == 0:
        inner_colors.append("#EBEBED")
    else:
        inner_colors.append("white")

alpha = 0.6
values_safe = np.maximum(values, 1e-6) ** alpha

widths = values_safe / values_safe.sum() * 2 * np.pi
angles = np.cumsum(widths) - widths / 2

base_radius = 1.5
step = 0.10

idx = np.arange(len(values))
total_radius = base_radius * (1 + step * idx)

outer_heights = np.full_like(total_radius, base_radius * 0.66)
inner_heights = total_radius - outer_heights

fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw=dict(polar=True))

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

ax.bar(
    angles,
    inner_heights,
    width=widths,
    bottom=0,
    color=inner_colors,
    edgecolor="white",
    linewidth=1.5
)

ax.bar(
    angles,
    outer_heights,
    width=widths,
    bottom=inner_heights,
    color=color_map,
    edgecolor="white",
    linewidth=1.5
)

label_radius = np.max(total_radius) * 1.65

for angle, feature_label in zip(angles, all_features):
    ax.text(
        angle,
        label_radius,
        feature_label,
        ha='center',
        va='center',
        fontsize=7
    )

ax.spines['polar'].set_visible(False)
ax.grid(False)
ax.set_yticklabels([])
ax.tick_params(axis='x', length=0)
ax.set_xticks([])

plt.tight_layout()

#    str(FIGURE5_OUTPUT_DIR / 'importance_Nightingale.png'),
#    dpi=300,
#    bbox_inches='tight'
#)

fig.savefig(FIGURE5_OUTPUT_DIR / 'figure05a_radial_importance.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Category shares shown in Figure 5a.
import matplotlib.pyplot as plt

categories = ['Development', 'Climate', 'Event']
values = df_grouped.set_index('Type').loc[['SE', 'ENV', 'CC'], 'Percentage (%)'].tolist()
colors = ['#579F9B', '#97D5CB', '#F5DF8B']

values_ratio = [v / 100 for v in values]

plt.figure(figsize=(6, 4), dpi=150)

plt.barh(categories, values_ratio, color=colors, height=0.75)

plt.xlim(0, 0.7)

plt.xticks([])

ax = plt.gca()
ax.set_yticklabels([])

ax.tick_params(
    axis='y',
    which='both',
    length=12,
    width=2.2,
    direction='in'
)

ax.tick_params(axis='x', length=0)

#for i, v in enumerate(values):
#    plt.text(values_ratio[i] + 0.01, i, f"{v:.1f}%", va='center')

plt.gca().invert_yaxis()

ax.spines['left'].set_linewidth(3.2)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)

plt.tight_layout()

plt.savefig(str(FIGURE5_OUTPUT_DIR / 'importance_Nightingale_legend.png'), dpi=300)

plt.show()